<a href="https://colab.research.google.com/github/IrynaKiriienko/Python_projects/blob/main/Portfolio_Project_2_ver_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [ ]:
# Аутентифікація
auth.authenticate_user()

# Створення клієнта для BigQuery
client = bigquery.Client(project="data-analytics-mate")

# SQL-запит
query = """

with session_info as (
    SELECT
        s.date,
        s.ga_session_id,
        sp.country,
        sp.device,
        sp.continent,
        sp.channel,
        ab.test,
        ab.test_group
    FROM `DA.ab_test` ab
    JOIN `DA.session` s
    ON ab.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` sp
    ON sp.ga_session_id = ab.ga_session_id
),
session_with_orders as (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct o.ga_session_id) as session_with_orders
    FROM `DA.order` o
    JOIN session_info
    ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),
events as (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        ep.event_name,
        count(ep.ga_session_id) as event_cnt
    FROM `DA.event_params` ep
    JOIN session_info
    ON ep.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        ep.event_name
),
session as (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(session_info.ga_session_id) as session_cnt
    FROM session_info
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),
accounts as (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count(distinct acs.ga_session_id) as new_account_cnt
    FROM `DA.account_session` acs
    JOIN session_info
    ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
)

SELECT
        session_with_orders.date,
        session_with_orders.country,
        session_with_orders.device,
        session_with_orders.continent,
        session_with_orders.channel,
        session_with_orders.test,
        session_with_orders.test_group,
        'session_with_orders' as event_name,
        session_with_orders.session_with_orders as value
FROM session_with_orders
UNION ALL
SELECT
        events.date,
        events.country,
        events.device,
        events.continent,
        events.channel,
        events.test,
        events.test_group,
        events.event_name,
        events.event_cnt as value
FROM events
UNION ALL
SELECT
        session.date,
        session.country,
        session.device,
        session.continent,
        session.channel,
        session.test,
        session.test_group,
        'session' as event_name,
        session.session_cnt as value
FROM session
UNION ALL
SELECT
        accounts.date,
        accounts.country,
        accounts.device,
        accounts.continent,
        accounts.channel,
        accounts.test,
        accounts.test_group,
        'new_account' as event_name,
        accounts.new_account_cnt as value
FROM accounts


"""

# Виконання запиту та виведення результату
query_job = client.query(query)
df = query_job.to_dataframe(create_bqstorage_client=False)
# df.to_excel('df_test.xlsx')
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-02,Paraguay,desktop,Americas,Organic Search,2,1,session_with_orders,1
1,2020-11-02,Paraguay,desktop,Americas,Organic Search,1,2,session_with_orders,1
2,2020-11-03,Ecuador,mobile,Americas,Direct,2,2,session_with_orders,1
3,2020-11-04,Slovenia,desktop,Europe,Organic Search,2,2,session_with_orders,1
4,2020-11-05,Serbia,desktop,Europe,Paid Search,1,2,session_with_orders,1


In [ ]:
# Перевіряю унікальні значення івентів для розрахунку
df["event_name"].unique()

array(['session_with_orders', 'new_account', 'session', 'user_engagement',
       'page_view', 'scroll', 'session_start', 'view_promotion',
       'view_item', 'add_payment_info', 'add_shipping_info',
       'first_visit', 'begin_checkout', 'add_to_cart',
       'view_search_results', 'select_promotion', 'select_item', 'click',
       'view_item_list'], dtype=object)

In [ ]:
# Розрахунки по тоталу
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new_account"]
denominator = "session"
df_results=[]

unique_test = df["test"].sort_values(ascending = True).unique()
test_values = list(unique_test) # + ["All tests"]

for test in test_values:
  for metric in metrics:
        denominator_control = df[(df["test"] == test) & (df["event_name"] == denominator) & (df["test_group"] == 1)]["value"].sum()
        count_control = df[(df["test"] == test) & (df["event_name"] == metric) & (df["test_group"] == 1)]["value"].sum()
        denominator_test = df[(df["test"] == test) & (df["event_name"] == denominator) & (df["test_group"] == 2)]["value"].sum()
        count_test = df[(df["test"] == test) & (df["event_name"] == metric) & (df["test_group"] == 2)]["value"].sum()

        control_rate = count_control / denominator_control if denominator_control > 0 else 0

        test_rate = count_test / denominator_test if denominator_test > 0 else 0

        if control_rate > 0:
                  changes_metric_in_percent = (test_rate - control_rate) / control_rate
        else:
                  changes_metric_in_percent = 0


        if denominator_test > 0 and denominator_control > 0:
                  z_stat, p_value = sm.stats.proportions_ztest([count_test, count_control], [denominator_test, denominator_control])
        else:
                  z_stat, p_value = 0, 1

        result = {
                          "level": "total",
                          "test_number": test,
                          "group_name": "test_total",
                          "parametrs": f'{metric} / {denominator}',
                          "metric": metric,
                          "denominator": denominator,
                          "value_metric_control": count_control,
                          "value_denominator_control": denominator_control,
                          "conversion_rate_control": control_rate,
                          "value_metric_test": count_test,
                          "value_denominator_test": denominator_test,
                          "conversion_rate": test_rate,
                          "metric_change": changes_metric_in_percent,
                          "z_stat": z_stat,
                          "p_value": p_value,
                          "is_significant": p_value < 0.05
                      }

        df_results.append(result)

df_results_total = pd.DataFrame(df_results)
df_results_total.to_csv('df_results_total.csv',  index=False, encoding='utf-8-sig')
df_results_total.head(40)

,level,test_number,group_name,parametrs,metric,denominator,value_metric_control,value_denominator_control,conversion_rate_control,value_metric_test,value_denominator_test,conversion_rate,metric_change,z_stat,p_value,is_significant
0,total,1,test_total,add_payment_info / session,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,0.125420,3.924884,0.000087,True
1,total,1,test_total,add_shipping_info / session,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,0.065605,2.603571,0.009226,True
2,total,1,test_total,begin_checkout / session,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,0.066606,2.978783,0.002894,True
3,total,1,test_total,new_account / session,new_account,session,3823,45362,0.084278,3681,45193,0.081451,-0.033543,-1.542883,0.122859,False
4,total,2,test_total,add_payment_info / session,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,0.035769,1.240994,0.214608,False
5,total,2,test_total,add_shipping_info / session,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,0.016510,0.709557,0.477979,False
6,total,2,test_total,begin_checkout / session,begin_checkout,session,4262,50637,0.084168,4313,50244,0.085841,0.019882,0.952898,0.340642,False
7,total,2,test_total,new_account / session,new_account,session,4165,50637,0.082252,4184,50244,0.083274,0.012419,0.588793,0.556000,False
8,total,3,test_total,add_payment_info / session,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,0.014746,0.643172,0.520112,False
9,total,3,test_total,add_shipping_info / session,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,-0.026212,-1.413727,0.157442,False


In [ ]:
df["country"].dropna().unique()


array(['Paraguay', 'Ecuador', 'Slovenia', 'Serbia', 'Malta', 'Croatia',
       'Kazakhstan', 'Qatar', 'Slovakia', 'North Macedonia',
       'New Zealand', 'Lithuania', 'Bahrain', 'Mongolia', 'Cambodia',
       'Azerbaijan', 'Latvia', 'Iraq', 'Belarus', 'Albania', 'Georgia',
       'Venezuela', 'Uruguay', 'Algeria', 'Kenya', 'Oman',
       'Myanmar (Burma)', 'Palestine', 'Ghana', 'Trinidad & Tobago',
       'Costa Rica', 'Dominican Republic', 'Tunisia', 'Kuwait',
       'Bulgaria', 'Puerto Rico', 'Guatemala', 'Lebanon', 'El Salvador',
       'Bolivia', 'Cyprus', 'Iceland', 'Panama', 'Jamaica', 'Armenia',
       'Bahamas', 'Nepal', 'Honduras', 'Estonia', 'Jordan', 'Macao',
       'Luxembourg', 'Kosovo', 'Bosnia & Herzegovina', '(not set)',
       'Argentina', 'Australia', 'Austria', 'Bangladesh', 'Belgium',
       'Brazil', 'Canada', 'Chile', 'China', 'Colombia', 'Czechia',
       'Denmark', 'Egypt', 'Finland', 'France', 'Germany', 'Greece',
       'Hong Kong', 'Hungary', 'India', 'Indon

In [ ]:
# Розрахунки по країнах, тестах, девайсу
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new_account"]
denominator = "session"
countries = ["United States", "United Kingdom", "Ukraine", "Taiwan", "Japan", "India", "China", "Brazil",  "Turkey", "Spain"]
df_results=[]
unique_test = df["test"].sort_values(ascending = True).unique()
# test_values = list(unique_test) + ["All tests"]

# Цикл для розрахунку по тестам
for test in unique_test:
  df_test = df[df["test"] == test]

# Цикл для розрахунку тестів по девайсам, по каналам та по країнам
  for country in countries:
          df_current_country = df_test[df_test["country"] == country]


# Цикл для розрахунку тестів по девайсам, по каналам та по країнам у розрізі метрик
          for metric in metrics:

# Без фільтрації по test
              denominator_control = df_current_country[(df_current_country["event_name"] == denominator) & (df_current_country["test_group"] == 1)]["value"].sum()
              count_control = df_current_country[(df_current_country["event_name"] == metric) & (df_current_country["test_group"] == 1)]["value"].sum()
              denominator_test = df_current_country[(df_current_country["event_name"] == denominator) & (df_current_country["test_group"] == 2)]["value"].sum()
              count_test = df_current_country[(df_current_country["event_name"] == metric) & (df_current_country["test_group"] == 2)]["value"].sum()

# Розраховуємо конверсію по контрольній та тестовій групам
              control_rate = count_control / denominator_control if denominator_control > 0 else 0
              test_rate = count_test / denominator_test if denominator_test > 0 else 0

    # Розраховуємо відсоток змін між тестовою та контрольною групами у відсотках
              if control_rate > 0:
                    changes_metric_in_percent = (test_rate - control_rate) / control_rate
              else:
                    changes_metric_in_percent = 0

    # Розраховуємо z-тест та p_value
              if denominator_test > 0 and denominator_control > 0:
                    z_stat, p_value = sm.stats.proportions_ztest([count_test, count_control], [denominator_test, denominator_control])
              else:
                    z_stat, p_value = 0, 1

    #Підсумовуємо результати розрахунків
              result = {
                            "level": "country",
                            "test_number": test,
                            "group_name": country,
                            "parametrs": f'{metric} / {denominator}',
                            "metric": metric,
                            "denominator": denominator,
                            "value_metric_control": count_control,
                            "value_denominator_control": denominator_control,
                            "conversion_rate_control": control_rate,
                            "value_metric_test": count_test,
                            "value_denominator_test": denominator_test,
                            "conversion_rate": test_rate,
                            "metric_change": changes_metric_in_percent,
                            "z_stat": z_stat,
                            "p_value": p_value,
                            "is_significant": p_value < 0.05
                        }

              df_results.append(result)

# Виконання запиту та виведення результату
df_results_countries = pd.DataFrame(df_results)
# df_results_countries.to_csv('df_results_by_countries.csv',  index=False, encoding='utf-8-sig')
df_results_countries.head()

,level,test_number,group_name,parametrs,metric,denominator,value_metric_control,value_denominator_control,conversion_rate_control,value_metric_test,value_denominator_test,conversion_rate,metric_change,z_stat,p_value,is_significant
0,country,1,United States,add_payment_info / session,add_payment_info,session,867,20010,0.043328,906,19681,0.046034,0.062451,1.304757,0.191976,False
1,country,1,United States,add_shipping_info / session,add_shipping_info,session,1375,20010,0.068716,1350,19681,0.068594,-0.001769,-0.047888,0.961806,False
2,country,1,United States,begin_checkout / session,begin_checkout,session,1713,20010,0.085607,1684,19681,0.085565,-0.000496,-0.015111,0.987944,False
3,country,1,United States,new_account / session,new_account,session,1758,20010,0.087856,1561,19681,0.079315,-0.097216,-3.073367,0.002117,True
4,country,1,United Kingdom,add_payment_info / session,add_payment_info,session,69,1439,0.047950,71,1450,0.048966,0.021179,0.127098,0.898863,False


In [ ]:
# Розрахунки по країнах, тестах, девайсу
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new_account"]
denominator = "session"
devices = df["device"].dropna().unique()
df_results=[]
unique_test = df["test"].sort_values(ascending = True).unique()
# test_values = list(unique_test) + ["All tests"]

# Цикл для розрахунку по тестам
for test in unique_test:
  df_test = df[df["test"] == test]
# Цикл для розрахунку тестів по девайсам
  for device in devices:
    df_device = df_test[df_test["device"] == device]




# Цикл для розрахунку тестів по девайсам, по каналам та по країнам у розрізі метрик
    for metric in metrics:

# Без фільтрації по test
              denominator_control = df_device[(df_device["event_name"] == denominator) & (df_device["test_group"] == 1)]["value"].sum()
              count_control = df_device[(df_device["event_name"] == metric) & (df_device["test_group"] == 1)]["value"].sum()
              denominator_test = df_device[(df_device["event_name"] == denominator) & (df_device["test_group"] == 2)]["value"].sum()
              count_test = df_device[(df_device["event_name"] == metric) & (df_device["test_group"] == 2)]["value"].sum()

# Розраховуємо конверсію по контрольній та тестовій групам
              control_rate = count_control / denominator_control if denominator_control > 0 else 0
              test_rate = count_test / denominator_test if denominator_test > 0 else 0

# Розраховуємо відсоток змін між тестовою та контрольною групами у відсотках
              if control_rate > 0:
                    changes_metric_in_percent = (test_rate - control_rate) / control_rate
              else:
                    changes_metric_in_percent = 0

# Розраховуємо z-тест та p_value
              if denominator_test > 0 and denominator_control > 0:
                    z_stat, p_value = sm.stats.proportions_ztest([count_test, count_control], [denominator_test, denominator_control])
              else:
                    z_stat, p_value = 0, 1

# Підсумовуємо результати розрахунків
              result = {
                            "level": "device",
                            "test_number": test,
                            "group_name": device,
                            "parametrs": f'{metric} / {denominator}',
                            "metric": metric,
                            "denominator": denominator,
                            "value_metric_control": count_control,
                            "value_denominator_control": denominator_control,
                            "conversion_rate_control": control_rate,
                            "value_metric_test": count_test,
                            "value_denominator_test": denominator_test,
                            "conversion_rate": test_rate,
                            "metric_change": changes_metric_in_percent,
                            "z_stat": z_stat,
                            "p_value": p_value,
                            "is_significant": p_value < 0.05
                        }

              df_results.append(result)

# Виконання запиту та виведення результату
df_results_devices = pd.DataFrame(df_results)
# df_results_devices.to_csv('df_results_devices.csv',  index=False, encoding='utf-8-sig')
df_results_devices.head()

,level,test_number,group_name,parametrs,metric,denominator,value_metric_control,value_denominator_control,conversion_rate_control,value_metric_test,value_denominator_test,conversion_rate,metric_change,z_stat,p_value,is_significant
0,device,1,desktop,add_payment_info / session,add_payment_info,session,1130,26467,0.042695,1256,26417,0.047545,0.113608,2.686998,0.007210,True
1,device,1,desktop,add_shipping_info / session,add_shipping_info,session,1711,26467,0.064647,1916,26417,0.072529,0.121932,3.586024,0.000336,True
2,device,1,desktop,begin_checkout / session,begin_checkout,session,2108,26467,0.079646,2404,26417,0.091002,0.142576,4.673980,0.000003,True
3,device,1,desktop,new_account / session,new_account,session,2203,26467,0.083236,2147,26417,0.081273,-0.023575,-0.821212,0.411526,False
4,device,1,mobile,add_payment_info / session,add_payment_info,session,810,17896,0.045262,942,17767,0.053020,0.171407,3.389330,0.000701,True


In [ ]:
# Розрахунки по країнах, тестах, девайсу
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new_account"]
denominator = "session"
channels = df["channel"].dropna().unique()
df_results=[]
unique_test = df["test"].sort_values(ascending = True).unique()
# test_values = list(unique_test) + ["All tests"]

# Цикл для розрахунку по тестам
for test in unique_test:
  df_test = df[df["test"] == test]


# Цикл для розрахунку тестів по девайсам та по каналам
  for channel in channels:
      df_channel = df_test[df_test["channel"] == channel]


# Цикл для розрахунку тестів по девайсам, по каналам та по країнам у розрізі метрик
      for metric in metrics:

# Без фільтрації по test
              denominator_control = df_channel[(df_channel["event_name"] == denominator) & (df_channel["test_group"] == 1)]["value"].sum()
              count_control = df_channel[(df_channel["event_name"] == metric) & (df_channel["test_group"] == 1)]["value"].sum()
              denominator_test = df_channel[(df_channel["event_name"] == denominator) & (df_channel["test_group"] == 2)]["value"].sum()
              count_test = df_channel[(df_channel["event_name"] == metric) & (df_channel["test_group"] == 2)]["value"].sum()

# Розраховуємо конверсію по контрольній та тестовій групам
              control_rate = count_control / denominator_control if denominator_control > 0 else 0
              test_rate = count_test / denominator_test if denominator_test > 0 else 0

    # Розраховуємо відсоток змін між тестовою та контрольною групами у відсотках
              if control_rate > 0:
                    changes_metric_in_percent = (test_rate - control_rate) / control_rate
              else:
                    changes_metric_in_percent = 0

    # Розраховуємо z-тест та p_value
              if denominator_test > 0 and denominator_control > 0:
                    z_stat, p_value = sm.stats.proportions_ztest([count_test, count_control], [denominator_test, denominator_control])
              else:
                    z_stat, p_value = 0, 1

    #Підсумовуємо результати розрахунків
              result = {
                            "level": "channel",
                            "test_number": test,
                            "group_name":channel,
                            "parametrs": f'{metric} / {denominator}',
                            "metric": metric,
                            "denominator": denominator,
                            "value_metric_control": count_control,
                            "value_denominator_control": denominator_control,
                            "conversion_rate_control": control_rate,
                            "value_metric_test": count_test,
                            "value_denominator_test": denominator_test,
                            "conversion_rate": test_rate,
                            "metric_change": changes_metric_in_percent,
                            "z_stat": z_stat,
                            "p_value": p_value,
                            "is_significant": p_value < 0.05
                        }

              df_results.append(result)

# Виконання запиту та виведення результату
df_results_channels = pd.DataFrame(df_results)
# df_results_channels.to_csv('df_results_channels.csv',  index=False, encoding='utf-8-sig')
df_results_channels.head()

,level,test_number,group_name,parametrs,metric,denominator,value_metric_control,value_denominator_control,conversion_rate_control,value_metric_test,value_denominator_test,conversion_rate,metric_change,z_stat,p_value,is_significant
0,channel,1,Organic Search,add_payment_info / session,add_payment_info,session,640,15675,0.040829,514,15631,0.032883,-0.194614,-3.730758,0.000191,True
1,channel,1,Organic Search,add_shipping_info / session,add_shipping_info,session,1021,15675,0.065136,922,15631,0.058985,-0.094422,-2.255094,0.024127,True
2,channel,1,Organic Search,begin_checkout / session,begin_checkout,session,1249,15675,0.079681,1144,15631,0.073188,-0.081489,-2.161956,0.030622,True
3,channel,1,Organic Search,new_account / session,new_account,session,1307,15675,0.083381,1301,15631,0.083232,-0.001789,-0.047745,0.961919,False
4,channel,1,Direct,add_payment_info / session,add_payment_info,session,392,10691,0.036666,516,10361,0.049802,0.358252,4.690261,0.000003,True


In [ ]:
df_combined = pd.concat([df_results_countries, df_results_devices, df_results_channels], ignore_index=True)
df_combined.to_csv('df_combined_witout total.csv',  index=False, encoding='utf-8-sig')
df_combined.head(50)

,level,test_number,group_name,parametrs,metric,denominator,value_metric_control,value_denominator_control,conversion_rate_control,value_metric_test,value_denominator_test,conversion_rate,metric_change,z_stat,p_value,is_significant
0,country,1,United States,add_payment_info / session,add_payment_info,session,867,20010,0.043328,906,19681,0.046034,0.062451,1.304757,0.191976,False
1,country,1,United States,add_shipping_info / session,add_shipping_info,session,1375,20010,0.068716,1350,19681,0.068594,-0.001769,-0.047888,0.961806,False
2,country,1,United States,begin_checkout / session,begin_checkout,session,1713,20010,0.085607,1684,19681,0.085565,-0.000496,-0.015111,0.987944,False
3,country,1,United States,new_account / session,new_account,session,1758,20010,0.087856,1561,19681,0.079315,-0.097216,-3.073367,0.002117,True
4,country,1,United Kingdom,add_payment_info / session,add_payment_info,session,69,1439,0.047950,71,1450,0.048966,0.021179,0.127098,0.898863,False
5,country,1,United Kingdom,add_shipping_info / session,add_shipping_info,session,85,1439,0.059069,98,1450,0.067586,0.144195,0.939737,0.347353,False
6,country,1,United Kingdom,begin_checkout / session,begin_checkout,session,99,1439,0.068798,110,1450,0.075862,0.102682,0.732852,0.463649,False
7,country,1,United Kingdom,new_account / session,new_account,session,122,1439,0.084781,112,1450,0.077241,-0.088932,-0.742682,0.457674,False
8,country,1,Ukraine,add_payment_info / session,add_payment_info,session,23,439,0.052392,20,465,0.043011,-0.179056,-0.662306,0.507775,False
9,country,1,Ukraine,add_shipping_info / session,add_shipping_info,session,32,439,0.072893,19,465,0.040860,-0.439449,-2.086302,0.036951,True


# Дашборд

#### https://public.tableau.com/app/profile/iryna.kiriienko/viz/abtestbychannelscountriesdevices/Dashboard1?publish=yes

#Файл з розрахунком по тоталу
#### https://drive.google.com/file/d/1MkYXrodAaN9TV2Tv6pNVxP5cMM0hOXbb/view?usp=sharing

#Файл з розрахунком в розрізі країн, девайсів та каналів
#### https://drive.google.com/file/d/1bbvERzqJmG6CZ-czQI96EZd-2cWnaCX-/view?usp=sharing

# Аналіз A/B тесту в тоталі

Було зроблено аналіз дашборду з фокусом на ключові метрики: конверсію (conversion_rate_control + metric_change), z_stat та p_value.
Тестування проводиться на рівні сесій для ключових етапів воронки: add_payment_info, add_shipping_info, begin_checkout, new_account. Розподіл трафіку можна вважати ідеально збалансований, бо на розподіл на групи становить 49-50%.
У першій тестовій групі по трьом показникам маємо p_value нижче 5%, а показники z_stat >1.96, отже цей тест можна взяти для подальших розрахунків. Другий та третій тести мають високі показники p_value та низькі показники z_stat. Четвертий тест має 2 показники, які мають значення p_value та z_stat у межах норми, для відхилення нульової гіпотези, але вони мають негативне значення у конверсії показників.
У розрізі тоталу ми бачимо зниження показників починаючи з етапу add_payment_info. Тобто користувачі додають інформацію про платіжні засоби, а потім не переходять до наступного кроку.

Така тендеція може бути через недостатність охоплення тестів, тому потрібно або продовжувати тест для збільшення кількості значень для кожного етапу воронки, або залучити більшу аудиторію для тестування.


# Аналіз в розрізі країн
Тест 1 — був успішним для Іспанії та Індії. Вони на нього відреагували найкраще.

Тест 2 – найкращі показники у США, Індії та Іспанії, однак для всіх країн завищені показники p_value та замалі z_stat. У Великобританії показники p_value та z_stat знаходяться у межах для відкидання нульової гіпотези, однак показники конверсії від’ємні, а отже ми не можемо прийняти цей тест. Найкращі показники у цьому тесті має Китай: за add_shipping_info, begin_checkout і трохи завищені показники p_value для add_payment_info.

Тест 3 – найгірші показники за всіма країнами. Тест не спрацював

Тест 4:
Великобританії, США, Іспанія та Китай не пройшли тест, хоч і є позитивні результати по деяким країнам у p_value, однак маємо негативну конверсію.
По Індії маємо хороший результат, тому можемо продовжувати тест по new_account для подальшого впровадження змін.

# Аналіз в розрізі каналів
Тест 1 є абсолютним лідером для залучення користувачів через Direct-канал. Статистично значуще зросли майже всі перші кроки.
По каналу «Undefined» бачимо стрімкий ріст конверсії і нульове значення p_value. Значення можуть бути або аномальними, або мати невелику вибірку.

Тест 2 знов таки показує дивні значення по Undefined. Серед інших каналів можна виділити Paid Search, який має позитивну конверсію та статистично значущі результати по add_shipping_info, begin_checkout. Показники по Social Search мають гарний старт на році add_payment info з подальшим падінням показників.

Тест 3 та Тест 4 провал по тестам. Рівень конверсії по всім каналам має негативні показники.

# Аналіз в розрізі девайссів

Тест 1 – бачимо гарну динаміку серед користувачів Desctop. Потрібно збільшити показники по new_account і можна впроваджувати зміни. Бачимо негативну конверсію серед користувачів Tablet, можливо потрібно покращити дизайн сторінок на цих пристроях.

Тест 2 не приніс очікуваних змін. Великого відтоку користувачів не спостерігаємо, однак зміни не принесли статистично значущих результатів.

Тест 3 знов таки провал по тесту. Великі значення p_value та від’ємна конверсія по всіх пристроях.

Тест 4 серед користувачів Desctop маємо статистично значущі зміни по всім категоріям, які призвели до негативної конверсії. Користувачі Tablet також не переходять до наступного кроку воронки, хоч і всі результати тестів є статистично значущими. А серед користувачів Mobile бачимо позитивну конверсію, однак тестування не призвело до бажаного ефекту.


# Загальний висновок

Тест 1 по всіх розрізах маємо дуже якісні статистично значущі результати серед користувачів Іспанії та Індії, які користуються десктопними пристроями та Direct каналом. Отже цей тест рекомендується для подальшого релізу.

Тест 2 готовий до релізу серед користувачів США по каналу Paid Search. Однак серед інших пристроїв та країн не відбулися статистично значущі зміни.

Тест 3 можна вважати глобальним провалом по всім метрикам та способам групування. Зміни не застосовуються.

Тест 4 Потрібно доопрацювання інтерфейсу під інші пристрої


В глобальному аналізі даних ми маємо як статистично значущі так і ні результати, а отже це є наслідком або неправильного обрання часу тестів (були якісь акції, розпродажі, значущі події) або недостатньої кількості вибірки, тому результати маємо неоднорідні і впроваджувати більшість змін не можна.
